# Valley Fever | Three-County Modeling (2015–2025)
**Maricopa + Pinal + Pima — one aligned county-month dataset.**

This notebook accepts the accompanying `VF_ThreeCounty_2015_2025_Master.xlsx`. You **do not need to redownload AZMet** or upload the original three county files.

The Excel already harmonizes PRISM, PM10 and AZMet variables; in particular, Maricopa wind speed was converted from mph to m/s. It contains 396 monthly source rows and 357 next-month forecasting rows.

**Research caution:** Historical event-date case counts may not be available immediately at the end of a month, so these are *retrospective* next-month forecast experiments, not yet an operational alert system. PRISM values come from county-centroid samples rather than geographic county-wide averages. 2025 is provisional. For 2024, retain all 12 months, and also report an April–December sensitivity result.

In [ ]:
# Cell 1 - install the packages Colab needs. No xlsxwriter dependency.
%pip -q install pandas numpy openpyxl scikit-learn

In [ ]:
# Cell 2 - upload the combined Excel once.
import os
from google.colab import files

NAME = 'VF_ThreeCounty_2015_2025_Master.xlsx'
if not os.path.exists(NAME):
    print('Upload VF_ThreeCounty_2015_2025_Master.xlsx')
    up = files.upload()
    if NAME not in up and not os.path.exists(NAME):
        raise FileNotFoundError(f'Expected {NAME}; uploaded: {list(up)}')
print('Ready:', NAME, f'{os.path.getsize(NAME):,} bytes')

In [ ]:
# Cell 3 - load the two key Excel sheets and validate our merge.
import numpy as np
import pandas as pd
import calendar
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

month = pd.read_excel(NAME, sheet_name='COUNTY_MONTH', dtype={'County_FIPS': str, 'Month': str})
forecast = pd.read_excel(NAME, sheet_name='FORECAST_READY',
                         dtype={'County_FIPS': str, 'Feature_Month': str, 'Target_Month': str})
expected_counties = {'Maricopa', 'Pinal', 'Pima'}
assert len(month) == 396, f'Expected 396 county-month rows; got {len(month)}'
assert len(forecast) == 357, f'Expected 357 forecast rows; got {len(forecast)}'
assert set(month['County']) == set(forecast['County']) == expected_counties
assert not month.duplicated(['County_FIPS', 'Month']).any(), 'Repeated county-month keys'
assert not forecast.duplicated(['County_FIPS', 'Target_Month']).any(), 'Repeated forecast targets'

month['County_FIPS'] = month['County_FIPS'].str.zfill(5)
forecast['County_FIPS'] = forecast['County_FIPS'].str.zfill(5)
expected_fips = {'Maricopa':'04013','Pinal':'04021','Pima':'04019'}
for c, fips in expected_fips.items():
    rows = month.loc[month['County'] == c]
    assert len(rows) == 132 and set(rows['County_FIPS']) == {fips}
    assert len(forecast.loc[forecast['County'] == c]) == 119

# Confirm each target truly matches the event-date case count of the following month.
lookup = month.set_index(['County_FIPS', 'Month'])['VF_cases_event_date']
for r in forecast.itertuples(index=False):
    target = int(lookup.loc[(r.County_FIPS, r.Target_Month)])
    assert int(r.Target_Cases_Next_Month) == target, (r.County, r.Target_Month)
    assert (pd.Period(r.Feature_Month, freq='M') + 1) == pd.Period(r.Target_Month, freq='M')

# Do not mix a future year's event-date outcome into the features for any target.
assert not any(c.startswith('Target_Month_') for c in forecast.columns)
print('PASS: 396 county-month records, 357 forecasts, 3 counties, unique keys and correct targets.')
print('Case totals by county / year:')
month['Year'] = month['Month'].str[:4].astype(int)
display(month.loc[month['Year'].isin([2024, 2025])].pivot_table(index='County', columns='Year', values='VF_cases_event_date', aggfunc='sum'))
print('Forecast splits:')
display(forecast.groupby(['Split','County']).size().unstack(fill_value=0))

In [ ]:
# Cell 4 - choose consistent predictors and build simple, leakage-aware models.
FEATURE_GROUPS = [
    'PRISM_rain_total_in', 'PRISM_tmean_F', 'EPA_PM10_mean_ugm3',
    'AZMET_soil4_C', 'AZMET_soil20_C', 'AZMET_RH_pct',
    'AZMET_wind_mean_ms', 'AZMET_wind_u_ms', 'AZMET_wind_v_ms', 'AZMET_VPD_kPa'
]
LAGS = ['current','lag1','lag2','lag3']
weather_cols = [f'{name}_{lag}' for name in FEATURE_GROUPS for lag in LAGS]
case_cols = ['VF_cases_current','VF_cases_lag1','VF_cases_lag2','Seasonal_Baseline_Cases']
input_cols = case_cols + weather_cols + ['month_sin','month_cos', 'County_Maricopa','County_Pinal','County_Pima']
missing_cols = sorted(set(case_cols + weather_cols) - set(forecast.columns))
assert not missing_cols, 'Missing predictors: '+str(missing_cols)

forecast['target_year'] = forecast['Target_Month'].str[:4].astype(int)
forecast['target_month_num'] = forecast['Target_Month'].str[5:7].astype(int)
forecast['month_sin'] = np.sin(2*np.pi*forecast['target_month_num']/12)
forecast['month_cos'] = np.cos(2*np.pi*forecast['target_month_num']/12)
for c in expected_counties:
    forecast[f'County_{c}'] = (forecast['County'] == c).astype(int)
for col in case_cols + weather_cols:
    forecast[col] = pd.to_numeric(forecast[col], errors='coerce')
forecast['Target_Cases_Next_Month'] = pd.to_numeric(forecast['Target_Cases_Next_Month'],errors='raise')

print('Missing predictor cells:', int(forecast[input_cols].isna().sum().sum()))
# Median filling occurs INSIDE the training pipeline, never using future evaluation data.
MODELS = {
  'Seasonal naive': None,
  'Ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=30)),
  'Nonlinear boosting': make_pipeline(SimpleImputer(strategy='median'),
      HistGradientBoostingRegressor(max_iter=110, learning_rate=.06,
                                   max_leaf_nodes=10,min_samples_leaf=15,random_state=42))
}

def predict_models(train, test, experiment):
    assert set(train['Target_Month']).isdisjoint(set(test['Target_Month']))
    assert train['Target_Month'].max() < test['Target_Month'].min(), 'Time leakage in split'
    Xtr, Xte = train[input_cols].to_numpy(dtype=float), test[input_cols].to_numpy(dtype=float)
    ytr = train['Target_Cases_Next_Month'].to_numpy(dtype=float)
    outputs=[]
    for name, estimator in MODELS.items():
        if estimator is None:
            prediction = test['Seasonal_Baseline_Cases'].to_numpy(dtype=float)
        else:
            model = clone(estimator)
            model.fit(Xtr, ytr)
            prediction = model.predict(Xte)
        prediction = np.maximum(0,np.asarray(prediction,dtype=float))
        for r,pred in zip(test.itertuples(index=False),prediction):
            outputs.append({'Experiment':experiment, 'Model':name, 'County':r.County,
                            'Target_Month':r.Target_Month, 'Actual_Cases':int(r.Target_Cases_Next_Month),
                            'Predicted_Cases':float(pred), 'Abs_Error':abs(float(pred)-float(r.Target_Cases_Next_Month)),
                            '2024_Early_Concern':r.Early_2024_Surveillance_Flag})
    return pd.DataFrame(outputs)

def metrics(df, period):
    res=[]
    for (model,county), g in df.groupby(['Model','County'],sort=True):
        y=g.Actual_Cases.to_numpy(dtype=float)
        p=g.Predicted_Cases.to_numpy(dtype=float)
        res.append({'Period':period,'Model':model,'County':county,'Months':len(g),
                    'MAE':round(mean_absolute_error(y,p),2),
                    'RMSE':round(np.sqrt(mean_squared_error(y,p)),2),
                    'WAPE_pct':round(100*np.abs(y-p).sum()/max(y.sum(),1),2)})
    return pd.DataFrame(res).sort_values(['County','MAE'])

In [ ]:
# Cell 5 - expanding-window development validation, year by year (2019–2023).
# No 2024 or 2025 target rows are used to select or validate models here.
parts=[]
for year in range(2019,2024):
    train = forecast.loc[forecast['target_year'] < year].copy()
    test  = forecast.loc[forecast['target_year'] == year].copy()
    parts.append(predict_models(train, test, f'Walkforward_{year}'))
walkforward = pd.concat(parts,ignore_index=True)
print('Walk-forward monthly forecast rows:',len(walkforward))
walkforward_metrics = metrics(walkforward,'2019-2023 development walk-forward')
display(walkforward_metrics)

In [ ]:
# Cell 6 - 2024 held-out test. Fit through Dec 2023 only, not 2024.
train_2024 = forecast.loc[forecast['target_year'] <= 2023].copy()
test_2024 = forecast.loc[forecast['target_year'] == 2024].copy()
result_2024 = predict_models(train_2024,test_2024,'2024 held-out')
print('Full 2024, all 12 target months:')
full_2024_metrics = metrics(result_2024,'2024 Jan-Dec')
display(full_2024_metrics)

print('2024 sensitivity check - April through December only (same trained models):')
late_2024 = result_2024.loc[result_2024['Target_Month']>='2024-04'].copy()
late_2024_metrics = metrics(late_2024, '2024 Apr-Dec only')
display(late_2024_metrics)

# Importantly, Jan–Mar remain in the original data and full-year evaluation.
assert len(result_2024)==3*3*12
assert len(late_2024)==3*3*9

In [ ]:
# Cell 7 - 2025 provisional evaluation is reported separately.
# Refit using observed target months through 2024; 2024 lab-count concerns remain a limitation.
train_2025 = forecast.loc[forecast['target_year'] <= 2024].copy()
test_2025 = forecast.loc[forecast['target_year'] == 2025].copy()
result_2025 = predict_models(train_2025,test_2025,'2025 provisional')
provisional_metrics = metrics(result_2025,'2025 provisional Jan-Dec')
display(provisional_metrics)

all_metrics = pd.concat([walkforward_metrics,full_2024_metrics,late_2024_metrics,provisional_metrics],ignore_index=True)
all_predictions = pd.concat([walkforward,result_2024,result_2025],ignore_index=True)
print('Retrospective evaluation complete. Compare per-county MAE/WAPE, not just pooled case counts.')

In [ ]:
# Cell 8 - export simple CSV files and a single ZIP. No xlsxwriter needed.
import zipfile
from pathlib import Path

export_files = {
  'VF_ThreeCounty_Model_Metrics.csv':all_metrics,
  'VF_ThreeCounty_Development_Walkforward.csv':walkforward,
  'VF_ThreeCounty_2024_Test.csv':result_2024,
  'VF_ThreeCounty_2024_AprDec_Sensitivity.csv':late_2024,
  'VF_ThreeCounty_2025_Provisional.csv':result_2025,
}
for name,df in export_files.items():
    df.to_csv(name,index=False)
notes = """THREE-COUNTY VALLEY FEVER RETROSPECTIVE MODEL RESULTS
Study: 2015-2025; feature month t -> target t+1.
Cases: ADHS event-date calendar-month data; 2025 provisional.
PRISM: county-centroid sample, not county-averaged raster.
AZMet: station-network weather; wind m/s harmonized from original Maricopa mph.
Walk-forward: predict each year 2019-2023 using preceding years only.
Held-out 2024: train through 2023; preserve January-March in full test.
Sensitivity: April-December 2024 errors using same 2024 held-out predictions.
2025: train through 2024; provisional evaluation, not model selection.
Reporting delay caveat: preceding-month event-date cases may not be available in real time.
Please avoid causal claims from observational environmental correlations.
"""
Path('READ_ME_FIRST.txt').write_text(notes,encoding='utf-8')
zip_name='VF_ThreeCounty_Model_Results.zip'
with zipfile.ZipFile(zip_name,'w',zipfile.ZIP_DEFLATED) as z:
    for f in list(export_files)+['READ_ME_FIRST.txt']:
        z.write(f)
print('Saved',zip_name,'with',len(export_files),'CSVs and methodology notes')
files.download(zip_name)

## What to look at next

1. In the **development walk-forward** table, identify models that perform consistently across all three counties.
2. Compare the same models on **2024 held-out** and separately **2024 April–December**. This is a sensitivity analysis, **not a correction to cases**.
3. Treat **2025 as provisional**. Do not select the best model solely by its performance here.
4. If a simple seasonal baseline wins in Pima or another county, keep that finding—it is scientifically useful.
5. Before publishing an operational prediction system, investigate the ADHS case-report delay and upgrade PRISM centroid values to county-area summaries.

**Potential next stage:** after reviewing Ridge and nonlinear boosting, extend to the planned Poisson/negative-binomial and tuned XGBoost models with pre-specified development validation and a preserved 2024 test.